In [1]:
#@title Imports

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import random
import plotly.express as px
import plotly.graph_objects as go


from collections import defaultdict
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from sklearn.metrics import jaccard_score

In [2]:
#@title Set Parameters
α = 0.7 # Weight for cosine similarity 0 (heavily jaccard weighted) to 1 (heavily cosine weighted)
β = 0.3 # Weight for review jaccard similarity
γ = 1 - α - β # Weight for favorite jaccard similarity

if (α + β > 1):
  raise ValueError("α + β must be less than or equal to 1")

In [5]:
#@title Load DF

df_base = pd.read_csv(r"C:\Users\osher\Desktop\stuf\schul\year 4\semester 2\needle in a data haystack\Milestone\data\filtered_user_anime.csv")
df = df_base.set_axis(["titleID", "userID", "review_score", "favorited"], axis=1)

# Pivot tables
review_matrix = df.pivot_table(index="titleID", columns="userID", values="review_score")
favorite_matrix = df.pivot_table(index="titleID", columns="userID", values="favorited")

# Fill for cosine (zeros for unrated)
review_matrix_cosine = review_matrix.fillna(0)

# Fill for Jaccard (binary: rated = 1, not rated = 0)
review_binary = review_matrix.notna().astype(int)
favorite_binary = favorite_matrix.fillna(0).astype(int)

df.describe()

,titleID,review_score,favorited
count,6.180237e+07,4.253134e+07,6.180237e+07
mean,2.167256e+04,8.001310e+00,3.447986e-02
std,1.340629e+04,1.569308e+00,1.824582e-01
min,1.000000e+00,1.000000e+00,0.000000e+00
25%,1.008700e+04,7.000000e+00,0.000000e+00
50%,2.327300e+04,8.000000e+00,0.000000e+00
75%,3.348600e+04,9.000000e+00,0.000000e+00
max,4.289700e+04,1.000000e+01,1.000000e+00


In [6]:
#@title Calculate Similarity Matrices

# Cosine similarity on reviews
cosine_sim = cosine_similarity(review_matrix_cosine)
cosine_sim_df = pd.DataFrame(cosine_sim, index=review_matrix.index, columns=review_matrix.index)

# Jaccard similarity function for binary matrices
def jaccard_matrix(binary_matrix):
    n = binary_matrix.shape[0]
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim[i, j] = jaccard_score(
                binary_matrix.iloc[i],
                binary_matrix.iloc[j],
                zero_division=0  # prevent warning and set undefined cases to 0
            )
    return pd.DataFrame(sim, index=binary_matrix.index, columns=binary_matrix.index)

# Jaccard on favorites
jaccard_fav_df = jaccard_matrix(favorite_binary)

# Jaccard on review presence
jaccard_review_df = jaccard_matrix(review_binary)



KeyboardInterrupt: 

In [ ]:
#@title Generate Heatmap

combined_sim_df = (
    α * cosine_sim_df +
    β * jaccard_review_df+
    γ * jaccard_fav_df
)



# === Load anime title mapping ===
anime_df = pd.read_csv("/content/drive/MyDrive/Haystack/anime.csv")
id_to_title = dict(zip(anime_df["MAL_ID"], anime_df["Name"]))

# === Copy matrix to avoid modifying original ===
sim_matrix = combined_sim_df.copy()

# === Distance + clustering ===
distance_matrix = 1 - sim_matrix
np.fill_diagonal(distance_matrix.values, 0)

linkage_matrix = linkage(squareform(distance_matrix), method="average")
order = leaves_list(linkage_matrix)

ordered_sim = sim_matrix.iloc[order, :].iloc[:, order]

# === Map titleIDs to names for plot only ===
x_labels = [id_to_title.get(i, str(i)) for i in ordered_sim.columns]
y_labels = [id_to_title.get(i, str(i)) for i in ordered_sim.index]

# === Plot interactive heatmap ===
fig = px.imshow(
    ordered_sim.values,
    labels=dict(x="", y="", color="Similarity"),
    x=x_labels,
    y=y_labels,
    color_continuous_scale="Viridis",
    zmin=0, zmax=1,
    aspect="equal"
)

# Clean ticks for large datasets
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

# Use full names in hover tooltips
fig.update_traces(
    hovertemplate="Anime X: %{x}<br>Anime Y: %{y}<br>Similarity: %{z:.2f}<extra></extra>"
)

fig.update_layout(
    title="Clustered Anime Similarity Matrix (Interactive)",
    width=800, height=800
)

fig.show()